<p>
<font size='5' face='Georgia, Arial'>IIC-2233 Apunte Programación Avanzada</font><br>
<font size='1'>&copy; 2015 Karim Pichara - Christian Pieringer. Todos los derechos reservados. Editado por Equipo Docente IIC2233 2018-1 al 2026-2</font>
</p>

# Tabla de contenidos

1. [Ejecución de código de ejemplo del material que usa PyQt](#Ejecución-de-código-de-ejemplo-del-material-que-usa-PyQt)
2. [Repaso: el *event loop*](#Repaso:-el-event-loop)
3. [Una GUI congelada](#Una-GUI-congelada)
4. [¿Y si usamos un *thread*?](#¿Y-si-usamos-un-thread?)
    1. [¿Por qué falla?](#¿Por-qué-falla?)
5. [La regla de oro](#La-regla-de-oro)
6. [¿Necesito un *thread*?](#¿Necesito-un-thread?)
7. [Debuggeo: el *hook* de excepciones](#Debuggeo:-el-hook-de-excepciones)
    1. [¿Por qué funciona?](#¿Por-qué-funciona?)

## Ejecución de código de ejemplo del material que usa PyQt

Jupyter no fue creado para ejecutar interfaces gráficas de escritorio, por lo que en este _notebook_ el código se muestra como texto y **no** se ejecuta. Cada ejemplo está en la carpeta `scripts/`, y te recomendamos **fuertemente** ejecutarlo desde tu computador e interactuar con él.

**Importante:** al ejecutar un _script_, la terminal debe estar posicionada en la misma carpeta del archivo `.py`. Por ejemplo:

```bash
cd scripts/1-event-loop
python3 1_gui_congelada.py
```

## Repaso: el *event loop*

La semana pasada vimos que `app.exec()` inicia el ***main loop*** o ***event loop*** de PyQt. Ahora es momento de entender bien qué hace, porque de eso depende todo lo que veremos esta semana.

El *event loop* es un ciclo que, mientras la aplicación esté abierta, repite lo siguiente:

1. Saca **un** evento de la cola de eventos: un click, una tecla presionada, una señal emitida, la orden de redibujar la ventana, etc.
2. Ejecuta el método (*slot*) conectado a ese evento.
3. Cuando ese método **termina**, vuelve al paso 1.

![Diagrama del event loop](img/event-loop.svg)

Lo importante es que **todo esto ocurre en un único _thread_**: el *thread* principal, que es el que ejecutó `app.exec()`. Todos los métodos conectados a señales de la interfaz, todos los `mousePressEvent`, `keyPressEvent`, y el propio dibujo de la ventana se ejecutan uno después de otro en ese mismo *thread*.

## Una GUI congelada

¿Qué pasa si uno de esos métodos se demora? Veamos el siguiente ejemplo, que tiene un botón cuyo *slot* tarda 5 segundos (usamos `sleep` para simular una tarea lenta, como esperar un mensaje por la red o hacer un cálculo pesado), y otro botón que solo cuenta *clicks*.

Este código se encuentra en el archivo `1-event-loop/1_gui_congelada.py`.

```python
import sys
from time import sleep
from PyQt6.QtWidgets import QApplication, QWidget, QLabel, QPushButton


class MiVentana(QWidget):
    def __init__(self) -> None:
        super().__init__()
        self.clicks = 0
        self.init_gui()

    def init_gui(self) -> None:
        # Configuramos los widgets de la interfaz
        self.label = QLabel("Status: esperando", self)
        self.boton_lento = QPushButton("Tarea lenta (5 segundos)", self)
        self.boton_contador = QPushButton("Clicks: 0", self)

        self.label.setGeometry(10, 10, 280, 30)
        self.boton_lento.setGeometry(10, 50, 280, 30)
        self.boton_contador.setGeometry(10, 90, 280, 30)

        self.boton_lento.clicked.connect(self.tarea_lenta)
        self.boton_contador.clicked.connect(self.contar_click)

        # Configuramos las propiedades de la ventana.
        self.setWindowTitle("GUI congelada")
        self.setGeometry(50, 50, 300, 130)
        self.show()

    def tarea_lenta(self) -> None:
        """
        Este método se ejecuta en el thread principal, el mismo que corre
        el event loop. Mientras duerme, el event loop no puede procesar
        ningún otro evento: ni clicks, ni redibujar la ventana.
        """
        self.label.setText("Status: trabajando...")  # ¿Alcanza a verse?
        sleep(5)
        self.label.setText("Status: tarea lenta terminada")

    def contar_click(self) -> None:
        self.clicks += 1
        self.boton_contador.setText(f"Clicks: {self.clicks}")


if __name__ == "__main__":

    def hook(type_, value, traceback):
        # Imprimimos el error con el hook original de Python. Al instalar
        # nuestro propio sys.excepthook, PyQt no cierra el programa
        sys.__excepthook__(type_, value, traceback)

    sys.excepthook = hook
    app = QApplication([])
    ventana = MiVentana()
    sys.exit(app.exec())
```

Ejecútalo, presiona "Tarea lenta" y luego intenta presionar varias veces el botón de *clicks*. Vas a notar tres cosas:

1. **La ventana no responde** durante 5 segundos: no puedes moverla, ni redimensionarla, y el sistema operativo puede incluso preguntarte si quieres forzar su cierre.
2. **El texto "Status: trabajando..." nunca aparece.** `setText` no dibuja inmediatamente: solo avisa que el *label* debe redibujarse, lo que es un evento más en la cola. Ese evento se procesa cuando `tarea_lenta` termina, y para entonces el texto ya cambió a "tarea lenta terminada".
3. **Los _clicks_ no se pierden**: quedan en la cola y, apenas termina la tarea lenta, se procesan todos juntos y el contador salta de golpe.

En resumen: **mientras un _slot_ se está ejecutando, el _event loop_ está detenido**. Por eso todo método que se ejecute en el *thread* principal debe terminar rápido.

## ¿Y si usamos un *thread*?

La solución natural, después de la semana de *threading*, es mover el trabajo lento a otro *thread*. Pero eso trae una nueva pregunta: si ese *thread* quiere mostrar algo en la interfaz, ¿puede modificar los *widgets* directamente?

Hagamos el experimento. El siguiente programa crea 100 *threads*, cada uno con su propio `QLabel` de un color al azar. Cada *thread* mueve **directamente** su *label*, llamando a `label.move()` cada 0,1 segundos.

Para crear los *threads* usamos `QThread`, la clase de *thread* de PyQt. Por ahora basta saber que se usa igual que `threading.Thread`: heredamos de ella, definimos `run` y la iniciamos con `start()`. La veremos en detalle en el *notebook* 3.

Este código se encuentra en el archivo `1-event-loop/2_sin_signal.py`.

```python
from random import randint
from time import sleep

from PyQt6.QtCore import QThread
from PyQt6.QtGui import QPixmap, QColor
from PyQt6.QtWidgets import QLabel, QWidget, QApplication
import sys


class Cuadrado(QThread):
    identificador = 0

    def __init__(self, label: QLabel, limite_x: int, limite_y: int):
        super().__init__()
        self.id = Cuadrado.identificador
        Cuadrado.identificador += 1

        # guardamos el label
        self.label = label

        # Seteamos la posición inicial y la guardamos para usarla como una property
        self._posicion = [0, 0]
        self.posicion = [randint(0, limite_x), randint(0, limite_y)]

    @property
    def posicion(self) -> list:
        return self._posicion

    # Cada vez que se actualicé la posición,
    # se actualiza la posición de la etiqueta
    @posicion.setter
    def posicion(self, nueva_posicion: list) -> None:
        self._posicion = nueva_posicion
        nuevo_x, nuevo_y = self.posicion
        self.label.move(nuevo_x, nuevo_y)

    def run(self) -> None:
        while True:
            sleep(0.1)
            nuevo_x = self.posicion[0] + randint(-2, 2)
            nuevo_y = self.posicion[1] + randint(-2, 2)
            self.posicion = [nuevo_x, nuevo_y]
```

La ventana (`MiVentana`) solo crea el fondo, los 100 *labels* y los 100 *threads*. Si ejecutas este código, lo más probable es que el programa **se caiga** con un error como `Segmentation fault (core dumped)`. Otras veces no se verá nada moviéndose, o funcionará un rato y luego colapsará. El resultado cambia en cada ejecución.

### ¿Por qué falla?

Los *widgets* de Qt **no son _thread-safe_**: están diseñados para ser usados únicamente desde el *thread* principal. Mientras un *thread* secundario modifica la posición de un *label*, el *event loop* puede estar al mismo tiempo dibujando ese mismo *label*, y el estado interno de Qt queda corrupto.

Ojo, este no es el típico problema de *race condition* que se arregla con un `Lock`: el *event loop* es código interno de Qt, que no sabe nada de nuestros *locks*.

## La regla de oro

> **Solo el _thread_ principal toca la interfaz. Los demás _threads_ le avisan mediante señales.**

¿Por qué las señales sí funcionan? Porque cuando un *thread* secundario emite una señal conectada a un método de un objeto que vive en el *thread* principal, PyQt **no ejecuta ese método en el _thread_ que emitió**. En vez de eso, pone un evento en la cola del *thread* principal (esto se llama una conexión *encolada*, o *queued connection*). Luego el *event loop* lo saca de la cola y ejecuta el método, igual que si fuera un *click*.

El siguiente código es el mismo experimento, pero ahora cada *thread* emite una señal con su nueva posición, y es la ventana quien mueve el *label*. Este código se encuentra en el archivo `1-event-loop/3_con_signal.py`.

```python
class Cuadrado(QThread):
    identificador = 0

    def __init__(self, senal_mover: pyqtSignal, limite_x: int, limite_y: int) -> None:
        super().__init__()
        self.id = Cuadrado.identificador
        Cuadrado.identificador += 1

        # guardamos la señal
        self.senal_mover = senal_mover

        # Seteamos la posición inicial y la guardamos para usarla como una property
        self._posicion = [0, 0]
        self.posicion = (randint(0, limite_x), randint(0, limite_y))

    @property
    def posicion(self) -> list:
        return self._posicion

    # Cada vez que se actualicé la posición,
    # se actualiza la posición de la etiqueta
    @posicion.setter
    def posicion(self, nueva_posicion: list) -> None:
        self._posicion = nueva_posicion
        nuevo_x, nuevo_y = self.posicion
        self.senal_mover.emit(self.id, nuevo_x, nuevo_y)

    def run(self):
        while True:
            sleep(0.1)
            nuevo_x = self.posicion[0] + randint(-2, 2)
            nuevo_y = self.posicion[1] + randint(-2, 2)
            self.posicion = (nuevo_x, nuevo_y)


class MiVentana(QWidget):
    senal_mover = pyqtSignal(int, int, int)

    def __init__(self) -> None:
        super().__init__()
        self.setWindowTitle("Correcto uso de señales")
        self.setGeometry(200, 200, 500, 500)

        # Definimos QLabel para el fondo de la ventana
        self.fondo = QLabel(self)
        self.fondo.setStyleSheet("background: orange")
        self.fondo.setGeometry(0, 0, 500, 500)

        self.cuadrados = []
        self.labels = {}

        for _ in range(100):
            self.crear_cuadrado()

        self.senal_mover.connect(self.mover)
        self.show()

    def crear_cuadrado(self) -> None:
        # Creamos el label y se lo pasamos al Cuadrado
        label = QLabel(self)
        label.setGeometry(-50, -50, 50, 50)

        # Creamos un QPixmap de color aleatorio
        pixmap = QPixmap(50, 50)
        pixmap.fill(QColor(randint(20, 200), randint(20, 200), randint(20, 200)))
        label.setPixmap(pixmap)
        label.show()

        nuevo_cuadrado = Cuadrado(self.senal_mover, self.width(), self.height())
        self.labels[nuevo_cuadrado.id] = label
        self.cuadrados.append(nuevo_cuadrado)
        nuevo_cuadrado.start()

    def mover(self, id: int, x: int, y: int) -> None:
        self.labels[id].move(x, y)
```

Al ejecutarlo, los 100 cuadrados se mueven sin problemas. Los *threads* siguen haciendo su trabajo (calcular la nueva posición y esperar), pero quien modifica la interfaz es siempre el *thread* principal, en el método `mover`.

## ¿Necesito un *thread*?

Antes de crear un *thread*, conviene preguntarse si realmente lo necesitas. Esta es la pregunta que guiará el resto de la semana:

| ¿Qué quiero hacer? | Ejemplo | Herramienta |
|---|---|---|
| Reaccionar a una acción del usuario, y la reacción es rápida | Cambiar un texto al hacer *click* | Un *slot* normal. Nada más. |
| Algo que se repite cada cierto tiempo, y cada repetición es rápida | Mover personajes, un reloj, una animación, un temporizador | `QTimer` (*notebook* 2). Sin *threads* y sin problemas de concurrencia. |
| Algo que **bloquea** o que tarda mucho | Esperar mensajes con `recv`, un cálculo de varios segundos | Un *thread* (`QThread` o `threading.Thread`) que avisa con señales (*notebooks* 3 y 4). |

Una buena señal de alerta es el `sleep`: si escribes un `sleep` dentro de un *slot*, vas a congelar la interfaz. Y si lo usas para "esperar hasta el siguiente paso" (por ejemplo, mover un personaje cada 50 ms), probablemente lo que quieres es un `QTimer`.

## Debuggeo: el *hook* de excepciones

En PyQt6, si ocurre una excepción no capturada dentro de un *slot*, el programa puede **cerrarse completamente** (`Aborted (core dumped)`), aunque el error haya sido menor. Que se cierre o no depende de tu sistema, como veremos más abajo. Para que el programa siga funcionando y solo imprima el error, podemos instalar nuestro propio *hook* de excepciones en `sys.excepthook`: si lo hacemos, PyQt lo llama a él en vez de cerrar el programa.

Todos los *scripts* de esta semana incluyen lo siguiente:

```python
if __name__ == "__main__":

    def hook(type_, value, traceback):
        # Imprimimos el error con el hook original de Python. Al instalar
        # nuestro propio sys.excepthook, PyQt no cierra el programa
        sys.__excepthook__(type_, value, traceback)

    sys.excepthook = hook
```

`sys.__excepthook__` guarda el *hook* original de Python, que es el que imprime el *traceback* completo. Nuestro `hook` simplemente lo llama.

### ¿Por qué funciona?

Si nuestro `hook` solo llama a la función original, ¿por qué el resultado es distinto? La clave es que **PyQt revisa si cambiaste `sys.excepthook`, no qué hace tu función**:

1. `sys.__excepthook__` es una copia de respaldo del *hook* original de Python, y `sys.excepthook` es el que Python usa de verdad. En un Python sin modificar (por ejemplo, el de Windows, macOS o la mayoría de las distribuciones Linux), al iniciar ambos son **la misma función**: `sys.excepthook is sys.__excepthook__` es `True`.
2. Una excepción dentro de un *slot* no tiene a dónde subir: quien llamó al *slot* fue el *event loop*, que es código C++ de Qt, no Python. Entonces PyQt atrapa la excepción y decide qué hacer.
3. Si `sys.excepthook` **sigue siendo el original**, PyQt asume que nadie se hace cargo de los errores: imprime el *traceback* y aborta el programa (`Aborted (core dumped)`). Lo hace a propósito, para que un error no pase desapercibido.
4. Si `sys.excepthook` **es otra función**, PyQt asume que tú te haces cargo: llama a tu función y devuelve el control al *event loop*, y el programa sigue.

Por lo mismo, un error en un *slot* **no cierra el programa aunque no instales nada**. PyQt ve que `sys.excepthook` ya no es el original y sigue de largo. En Windows o macOS, en cambio, el mismo código sí se cae. Por eso igual instalamos nuestro `hook` en todos los *scripts*: así el comportamiento es el mismo en cualquier computador.

El cierre del programa no está dentro de `sys.__excepthook__` (que solo imprime): lo agrega PyQt cuando ve que el *hook* es el original. Se puede comprobar con dos instalaciones que imprimen exactamente lo mismo:

| Instalación | Resultado ante un error en un *slot* |
|---|---|
| `sys.excepthook = sys.__excepthook__` (la misma función original) | El programa se cae con `Aborted (core dumped)` |
| `sys.excepthook = hook`, donde `hook` solo llama a `sys.__excepthook__` | Se imprime el error y el programa sigue funcionando |

Ojo: que el programa siga funcionando no significa que haya quedado bien. El *slot* que falló se interrumpió a la mitad, así que puede haber dejado datos o la interfaz en un estado inconsistente. El *hook* sirve para ver el error y seguir probando, no para ignorarlo.